
# MCP Server — Authentication (with & without DCR) and Authorization (Eunomia)

This end-to-end notebook shows **how to secure a FastMCP server** with:
- **Authentication (non‑DCR)** via **JWT Bearer** and **OAuth Proxy** (GitHub/Google/Azure-style providers).
- **Authentication (DCR)** via **Remote OAuth** (providers that support **Dynamic Client Registration** such as WorkOS AuthKit/Descope).
- **Authorization** via **Eunomia** middleware to control what tools/resources/prompts can be **listed** and **executed**.

It also includes **cells that export runnable scripts** for servers, clients, and policy files so you can run them outside the notebook.



> **Docs used (highly recommended reading):**
> - FastMCP Authentication Overview, Token Verification, Remote OAuth, OAuth Proxy citeturn1view0turn2view0turn3view0turn3view1  
> - Eunomia Authorization middleware for FastMCP citeturn0view0
> - Client auth helpers (Bearer, OAuth) citeturn4search1turn4search17



## Prerequisites

> Run these locally (the execution environment here has no internet access).

```bash
# Core
pip install fastmcp

# Authorization (policy engine)
pip install eunomia-mcp

# Optional (GitHub/Google/Azure providers behind OAuth Proxy)
# Nothing extra required beyond fastmcp; proxy config uses your provider's OAuth app

# Optional: type hints / dev tools
pip install uvicorn python-dotenv
```



## Project layout

This notebook can **export** the following runnable files:

```
server_token_verification.py   # Non-DCR: JWT verification (JWKS/HMAC/static key) 
server_oauth_proxy.py          # Non-DCR: OAuth Proxy (e.g., GitHub/Google/Azure)
server_remote_oauth.py         # DCR: Remote OAuth (e.g., WorkOS AuthKit/Descope)
mcp_policies.json              # Eunomia policy file (generated via CLI or template here)
client_bearer.py               # Client using Bearer token
client_oauth.py                # Client using OAuth helper and browser flow
```

You can run them with `python <script>.py` once you’ve configured env vars or in-code settings.



## 1) Minimal FastMCP server (no auth) — for context

This is just to show the starting point. We won't export this one.


In [ ]:

# Minimal illustrative server; not exported.
# from fastmcp import FastMCP
#
# mcp = FastMCP("Demo Server")
#
# @mcp.tool()
# def add(a: int, b: int) -> int:
#     """Add two numbers"""
#     return a + b
#
# if __name__ == "__main__":
#     mcp.run()  # default SSE transport



## 2) Authentication (non‑DCR) — **Token Verification (JWTVerifier / StaticTokenVerifier)**

Use this when you **already issue tokens elsewhere** (API Gateway / Entra ID / Auth0 JWTs / internal signer).  
FastMCP validates tokens — via **JWKS**, **HMAC**, or **static public key**, and **does not expose OAuth discovery** to clients.  
Best fit for **internal systems** and **CI/CD**. citeturn2view0


In [1]:

%%writefile mcpserver/server_token_verification.py
from fastmcp import FastMCP
from fastmcp.server.auth.providers.jwt import JWTVerifier, StaticTokenVerifier

# === Option A: JWKS / asymmetric verification (PROD-friendly) ===
# verifier = JWTVerifier(
#     jwks_uri="https://auth.example.com/.well-known/jwks.json",
#     issuer="https://auth.example.com",
#     audience="mcp-production-api",
# )
#
# === Option B: HMAC / symmetric shared secret (internal) ===
# verifier = JWTVerifier(
#     public_key="your-32+char-shared-secret",
#     issuer="internal-auth",
#     audience="mcp-internal-api",
#     algorithm="HS256",   # HS256 / HS384 / HS512
# )
#
# === Option C: Static public key (dev/test) ===
# verifier = JWTVerifier(
#     public_key="""-----BEGIN PUBLIC KEY-----
# YOUR_PEM_KEY
# -----END PUBLIC KEY-----""",
#     issuer="https://auth.example.com",
#     audience="mcp-dev-api",
# )

# === Option D: Static tokens (DEV-ONLY) ===
verifier = StaticTokenVerifier(
    tokens={
        "dev-alice-token": {"client_id": "alice@company.com", "scopes": ["read:data", "write:data"]},
        "dev-guest-token": {"client_id": "guest", "scopes": ["read:data"]},
    },
    required_scopes=["read:data"],
)

mcp = FastMCP(name="Protected Server (Token Verification)", auth=verifier)

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

if __name__ == "__main__":
    mcp.run(transport='streamable-http')  # Run SSE/HTTP; include Authorization: Bearer <token> from your client


Writing mcpserver/server_token_verification.py



### Test client (Bearer token)

The FastMCP client can send a **Bearer** token by setting `auth="<access_token>"` (do **not** prefix with `Bearer `; the client does that). citeturn4search1


In [3]:

%%writefile mcpserver/client_bearer.py
import asyncio
from fastmcp.client import Client

TOKEN = "dev-alice-token"  # For StaticTokenVerifier above; replace with a real JWT in production

async def main():
    async with Client("http://localhost:8000/mcp", auth=TOKEN) as client:
        # list tools
        tools = await client.list_tools()
        print("TOOLS:", [t.name for t in tools])

        # call a tool
        res = await client.call_tool("add", arguments={"a": 2, "b": 3})
        print("add(2,3) ->", res)

if __name__ == "__main__":
    asyncio.run(main())


Overwriting mcpserver/client_bearer.py



## 3) Authentication (non‑DCR) — **OAuth Proxy** (GitHub/Google/Azure, etc.)

Use this when your provider **does not support DCR**. OAuth Proxy **presents MCP‑compatible discovery and dynamic callbacks to clients**, while using your **pre‑registered OAuth app** (fixed redirect). Configure the proxy with your provider **Authorization URL**, **Token URL**, and **client credentials**. citeturn3view1


In [ ]:

# %%writefile server_oauth_proxy.py
from fastmcp import FastMCP
from fastmcp.server.auth import OAuthProxy
from fastmcp.server.auth.providers.jwt import JWTVerifier

# Token verifier for your provider's JWTs (often via OIDC/JWKS)
token_verifier = JWTVerifier(
    jwks_uri="https://provider.example.com/.well-known/jwks.json",
    issuer="https://provider.example.com",
    audience="your-app-id",  # set to your OAuth app audience if required
)

auth = OAuthProxy(
    upstream_authorization_endpoint="https://provider.example.com/oauth/authorize",
    upstream_token_endpoint="https://provider.example.com/oauth/token",
    upstream_client_id="YOUR_CLIENT_ID",
    upstream_client_secret="YOUR_CLIENT_SECRET",
    token_verifier=token_verifier,
    base_url="http://localhost:8000",           # your server's public/base URL
    # redirect_path="/auth/callback",           # optional; default shown here
    # allowed_client_redirect_uris=["http://localhost:*", "https://*.yourcompany.com/*"],
    # extra_authorize_params={"audience": "https://api.example.com"},  # provider-specific
)

mcp = FastMCP(name="OAuth Proxy Server", auth=auth)

@mcp.tool()
def whoami() -> str:
    """Return the subject from token claims (if available)."""
    from fastmcp.server.context import request_context
    access = request_context().access  # token claims/identity if verified
    return str(access.claims if access else {})

if __name__ == "__main__":
    mcp.run()



### Client (OAuth — browser flow)

For user-interactive OAuth, use the **client OAuth helper**, which opens a browser and runs a local callback server. citeturn4search17


In [ ]:

# %%writefile client_oauth.py
import asyncio
from fastmcp.client import Client
from fastmcp.client.auth import OAuth  # opens browser + local callback

async def main():
    async with Client("http://localhost:8000/mcp", auth=OAuth()) as client:
        tools = await client.list_tools()
        print("TOOLS:", [t.name for t in tools.tools])
        res = await client.call_tool("whoami", arguments={})
        print("whoami ->", res)

if __name__ == "__main__":
    asyncio.run(main())



## 4) Authentication (with **DCR**) — **Remote OAuth (RemoteAuthProvider)**

Use this when your identity provider **supports Dynamic Client Registration (DCR)** (e.g., **WorkOS AuthKit**, **Descope**). Clients can **self‑register** and fetch tokens **automatically** using MCP discovery at `/.well-known/oauth-protected-resource`. citeturn3view0


In [ ]:

# %%writefile server_remote_oauth.py
from fastmcp import FastMCP
from fastmcp.server.auth.providers.workos import AuthKitProvider  # example DCR provider

auth = AuthKitProvider(
    authkit_domain="https://YOUR_PROJECT.authkit.app",  # your AuthKit project domain
    base_url="http://localhost:8000",                   # public/base URL of your MCP server
)

mcp = FastMCP(name="Remote OAuth (DCR) Server", auth=auth)

@mcp.tool()
def echo(text: str) -> str:
    """Echo text back to the caller"""
    return text

if __name__ == "__main__":
    mcp.run()



### Environment‑only configuration (12‑factor friendly)

FastMCP supports **environment variable** config for providers (e.g., GitHub/Google/JWT/WorkOS) so you can avoid hardcoding secrets. See the **Authentication** page for `FASTMCP_SERVER_AUTH` and provider‑specific envs. citeturn1view0



## 5) Authorization — **Eunomia middleware**

Eunomia plugs in as a **FastMCP middleware** that:
- **Filters listings** (tools/resources/prompts) based on policy.
- **Blocks executions** (tool calls/resource reads) not allowed by policy.
- Supports **embedded** (default) or **remote** policy decision service.
Add it to **any** of the above servers. citeturn0view0


In [4]:

%%writefile mcpserver/mcp_with_eunomia.py
from fastmcp import FastMCP
from eunomia_mcp import create_eunomia_middleware

mcp = FastMCP("Secure MCP with Eunomia")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def secret_op() -> str:
    """Highly restricted op"""
    return "top secret"

# Attach middleware (use embedded Eunomia by default)
middleware = create_eunomia_middleware(policy_file="mcp_policies.json")
mcp.add_middleware(middleware)

if __name__ == "__main__":
    mcp.run(transport='streamable-http')


Writing mcpserver/mcp_with_eunomia.py



### Policy file (quick template)

You can generate a starter policy with the Eunomia CLI:

```bash
eunomia-mcp init                # default (discovers mcp object)
# or
eunomia-mcp init --custom-mcp "mcp_with_eunomia:mcp"
eunomia-mcp validate mcp_policies.json
```

Below is a **minimal** illustrative policy that *hides* `secret_op` and only allows `add` for specific subjects.


In [6]:

%%writefile mcpserver/mcp_policies.json
{
  "version": "1.0",
  "name": "mcp-default-policy",
  "description": "Default policy for a MCP server",
  "rules": [
    {
      "name": "unrestricted-access",
      "description": "All principals can list and execute tools, resources, and prompts",
      "effect": "allow",
      "principal_conditions": [],
      "resource_conditions": [{"path":"attributes.name","operator":"in","value":["add","secret_op"]}],
      "actions": [
        "list","execute"
      ]
    }
  ],
  "default_effect": "deny"
}

Overwriting mcpserver/mcp_policies.json



## 6) End‑to‑end testing checklist

1. **Token Verification (dev static):**  
   - Run: `python server_token_verification.py`  
   - Client: `python client_bearer.py` (with `TOKEN=dev-alice-token`) → `add` should work; missing/invalid token should 401/403. citeturn2view0

2. **OAuth Proxy (non‑DCR):**  
   - Register app at provider; set redirect URI `http://localhost:8000/auth/callback`.  
   - Fill `server_oauth_proxy.py` with provider endpoints + credentials; run it.  
   - Run `python client_oauth.py` → browser opens → login/consent → client connects and calls `whoami`. citeturn3view1turn4search17

3. **Remote OAuth (DCR):**  
   - Configure `AuthKitProvider` (or equivalent) with your project domain and `base_url`.  
   - Run `server_remote_oauth.py` and `client_oauth.py` → auto‑registration + login flow. citeturn3view0

4. **Eunomia Authorization:**  
   - Run `python mcp_with_eunomia.py`.  
   - With a token **without** `tools:add` → `add` should be hidden/blocked.  
   - With a token **with** `tools:add` → `add` visible and callable; `secret_op` remains hidden/blocked. citeturn0view0



## 7) Security notes & tips

- Prefer **JWKS** for production (key rotation) and **avoid static tokens** outside dev. citeturn2view0  
- For **non‑DCR** providers (GitHub/Google/Azure), use **OAuth Proxy**. For **DCR** providers (WorkOS AuthKit/Descope), use **Remote OAuth**. citeturn1view0  
- Lock down `allowed_client_redirect_uris` for your environments (dev vs prod) when using OAuth Proxy. citeturn3view1  
- Keep **authorization** (Eunomia policies) separate from **authentication**; use token **claims** (subject, scopes, roles) in policy rules. citeturn0view0
